# Real-Time Fraud / Anomaly Detection System

Architecture
------------
1. Feature Engineering
   - Transaction-level features (amount, hour, merchant category)
   - Behavioral/velocity features (spending vs the user's own historical average,
     transactions in the last hour) — velocity features are the single most
     important signal class in real fraud systems, since fraud usually shows up
     as a sudden deviation from a user's normal pattern, not an absolute value

2. Two-model ensemble (standard real-world pattern for fraud):
   a. SupervisedFraudClassifier (XGBoost)
      - Learns from historically labeled fraud/not-fraud transactions
      - Strong when enough labeled fraud examples exist
   b. UnsupervisedAnomalyDetector (Isolation Forest)
      - Learns "normal" transaction behavior without needing fraud labels
      - Catches novel fraud patterns the supervised model has never seen labeled
        examples of (critical because fraud patterns evolve to evade known rules)
   - Final score = weighted blend of both signals — this mirrors real production
     fraud stacks (e.g. a supervised score + an anomaly score feeding a final
     decision layer), since relying on labels alone misses emerging fraud tactics

3. StreamingFraudPipeline
   - Simulates processing transactions one at a time (as they'd arrive from a
     Kafka/Kinesis stream) instead of a batch
   - Applies a decision policy: BLOCK / REVIEW / ALLOW based on score thresholds
     — this three-tier policy (not just binary) reflects real systems, where high
     confidence blocks automatically, moderate scores go to human review queues,
     and low scores pass through

4. Evaluation
   - Precision-Recall AUC (more informative than ROC-AUC under heavy class
     imbalance, which is always the case in fraud — typically <1% of transactions)
   - Confusion matrix at the operating threshold used by the decision policy

In [ ]:
#importing libraries
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, confusion_matrix, classification_report
)
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier

In [ ]:
# ----------------------------------------------------------------------------
# 1. Synthetic transaction stream (stand-in for a real payments event log)
# ----------------------------------------------------------------------------
def generate_synthetic_data(n_users=1000, n_transactions=40000, fraud_rate=0.015, seed=42):
    rng = np.random.default_rng(seed)

    user_ids = rng.integers(0, n_users, size=n_transactions)
    # Each user has a typical spend level (their "normal" behavior baseline)
    user_avg_spend = rng.gamma(shape=2.0, scale=40, size=n_users)

    hour_of_day = rng.integers(0, 24, size=n_transactions)
    merchant_category = rng.choice(
        ["grocery", "electronics", "travel", "restaurant", "online_retail", "gas_station"],
        size=n_transactions
    )
    is_fraud = (rng.random(n_transactions) < fraud_rate).astype(int)

    amounts = np.zeros(n_transactions)
    txns_last_hour = rng.poisson(1.0, size=n_transactions)

    for i in range(n_transactions):
        base = user_avg_spend[user_ids[i]]
        if is_fraud[i]:
            # Fraud: amount often spikes above the user's normal spend, and tends to
            # cluster at odd hours with higher recent transaction velocity — but with
            # enough overlap/noise vs normal behavior to mimic a realistic, imperfectly
            # separable signal (real fraud is not perfectly distinguishable either).
            amounts[i] = base * rng.uniform(1.5, 6)
            hour_of_day[i] = rng.choice(range(24))
            if rng.random() < 0.6:
                hour_of_day[i] = rng.choice([1, 2, 3, 4, 23])
            txns_last_hour[i] = rng.poisson(2.5)
        else:
            amounts[i] = max(1, rng.normal(base, base * 0.5))
            # small fraction of legit transactions are naturally high-spend outliers,
            # to avoid a trivially separable synthetic dataset
            if rng.random() < 0.03:
                amounts[i] *= rng.uniform(2, 4)

    df = pd.DataFrame({
        "transaction_id": np.arange(n_transactions),
        "user_id": user_ids,
        "amount": amounts,
        "hour_of_day": hour_of_day,
        "merchant_category": merchant_category,
        "txns_last_hour": txns_last_hour,
        "is_fraud": is_fraud,
    })
    df["user_avg_spend"] = df.user_id.map(pd.Series(user_avg_spend))
    return df


def engineer_features(df):
    df = df.copy()
    df["amount_to_avg_ratio"] = df.amount / (df.user_avg_spend + 1)
    df["is_odd_hour"] = df.hour_of_day.isin([0, 1, 2, 3, 4, 23]).astype(int)
    df["high_velocity"] = (df.txns_last_hour >= 3).astype(int)
    df = pd.get_dummies(df, columns=["merchant_category"], drop_first=True)
    return df


FEATURE_COLS_BASE = ["amount", "hour_of_day", "txns_last_hour", "amount_to_avg_ratio",
                      "is_odd_hour", "high_velocity"]


In [ ]:
# ----------------------------------------------------------------------------
# 2a. Supervised classifier
# ----------------------------------------------------------------------------
class SupervisedFraudClassifier:
    def __init__(self, seed=42):
        self.seed = seed

    def fit(self, X, y):
        neg, pos = (y == 0).sum(), (y == 1).sum()
        self.model = XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=neg / pos, eval_metric="aucpr", random_state=self.seed,
        )
        self.model.fit(X, y)
        return self

    def score(self, X):
        return self.model.predict_proba(X)[:, 1]

In [ ]:
# ----------------------------------------------------------------------------
# 2b. Unsupervised anomaly detector
# ----------------------------------------------------------------------------
class UnsupervisedAnomalyDetector:
    def __init__(self, contamination=0.02, seed=42):
        self.model = IsolationForest(contamination=contamination, random_state=seed, n_estimators=200)

    def fit(self, X):
        self.model.fit(X)
        return self

    def score(self, X):
        # decision_function: higher = more normal. Flip + normalize so higher = more anomalous.
        raw = -self.model.decision_function(X)
        return raw

In [ ]:
# ----------------------------------------------------------------------------
# 3. Ensemble fraud engine
# ----------------------------------------------------------------------------
class FraudDetectionEngine:
    def __init__(self, feature_cols, alpha=0.7, seed=42):
        """alpha: weight on supervised score vs unsupervised anomaly score."""
        self.feature_cols = feature_cols
        self.alpha = alpha
        self.supervised = SupervisedFraudClassifier(seed=seed)
        self.anomaly = UnsupervisedAnomalyDetector(seed=seed)
        self.scaler_sup = MinMaxScaler()
        self.scaler_anom = MinMaxScaler()

    def fit(self, df):
        X = df[self.feature_cols]
        y = df["is_fraud"]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        self.supervised.fit(self.X_train, self.y_train)
        self.anomaly.fit(self.X_train)  # trained on all data (unsupervised — doesn't need labels)

        sup_scores = self.supervised.score(self.X_train).reshape(-1, 1)
        anom_scores = self.anomaly.score(self.X_train).reshape(-1, 1)
        self.scaler_sup.fit(sup_scores)
        self.scaler_anom.fit(anom_scores)
        return self

    def score(self, X):
        sup = self.scaler_sup.transform(self.supervised.score(X).reshape(-1, 1)).flatten()
        anom = self.scaler_anom.transform(self.anomaly.score(X).reshape(-1, 1)).flatten()
        return self.alpha * sup + (1 - self.alpha) * np.clip(anom, 0, 1)

    def evaluate(self):
        scores = self.score(self.X_test)
        ap = average_precision_score(self.y_test, scores)
        print(f"Average Precision (PR-AUC): {ap:.4f}")

        precision, recall, thresholds = precision_recall_curve(self.y_test, scores)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        best_idx = np.argmax(f1)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        print(f"Best F1-optimal threshold: {best_threshold:.3f} (F1={f1[best_idx]:.3f})")

        preds = (scores >= best_threshold).astype(int)
        print("\nClassification report at best threshold:")
        print(classification_report(self.y_test, preds, digits=3))
        print("Confusion matrix:\n", confusion_matrix(self.y_test, preds))
        return ap, best_threshold

In [ ]:
# ----------------------------------------------------------------------------
# 4. Real-time decision policy (BLOCK / REVIEW / ALLOW)
# ----------------------------------------------------------------------------
class DecisionPolicy:
    def __init__(self, block_threshold=0.85, review_threshold=0.5):
        self.block_threshold = block_threshold
        self.review_threshold = review_threshold

    def decide(self, score):
        if score >= self.block_threshold:
            return "BLOCK"
        elif score >= self.review_threshold:
            return "REVIEW"
        return "ALLOW"

In [ ]:
# ----------------------------------------------------------------------------
# 5. Streaming simulation — processes transactions one at a time
# ----------------------------------------------------------------------------
class StreamingFraudPipeline:
    def __init__(self, engine, policy):
        self.engine = engine
        self.policy = policy

    def process_stream(self, transactions_df, feature_cols, max_events=15, simulate_latency=False):
        print(f"{'txn_id':>8} {'amount':>10} {'score':>8}  decision")
        print("-" * 45)
        for _, row in transactions_df.head(max_events).iterrows():
            X = transactions_df.loc[[row.name], feature_cols]
            score = self.engine.score(X)[0]
            decision = self.policy.decide(score)
            print(f"{int(row.transaction_id):>8} {row.amount:>10.2f} {score:>8.3f}  {decision}"
                  + ("  <-- ACTUAL FRAUD" if row.is_fraud else ""))
            if simulate_latency:
                time.sleep(0.02)


if __name__ == "__main__":
    raw_df = generate_synthetic_data(n_users=1000, n_transactions=40000, fraud_rate=0.015)
    features_df = engineer_features(raw_df)

    feature_cols = FEATURE_COLS_BASE + [c for c in features_df.columns if c.startswith("merchant_category_")]

    engine = FraudDetectionEngine(feature_cols, alpha=0.7).fit(features_df)
    print("=== Fraud Detection Engine Evaluation ===")
    ap, threshold = engine.evaluate()

    policy = DecisionPolicy(block_threshold=max(threshold + 0.15, 0.8), review_threshold=threshold)
    pipeline = StreamingFraudPipeline(engine, policy)

    print("\n=== Simulated Real-Time Transaction Stream (first 15 test transactions, sorted to show a mix) ===")
    demo_df = features_df.loc[engine.X_test.index].copy()
    demo_df["is_fraud"] = raw_df.loc[engine.X_test.index, "is_fraud"]
    # Interleave a few known-fraud rows so the demo output is illustrative
    fraud_rows = demo_df[demo_df.is_fraud == 1].head(5)
    normal_rows = demo_df[demo_df.is_fraud == 0].head(10)
    demo_sample = pd.concat([fraud_rows, normal_rows]).sample(frac=1, random_state=1)

    pipeline.process_stream(demo_sample, feature_cols, max_events=15)


=== Fraud Detection Engine Evaluation ===
Average Precision (PR-AUC): 0.7017
Best F1-optimal threshold: 0.829 (F1=0.696)

Classification report at best threshold:
              precision    recall  f1-score   support

           0      0.995     0.995     0.995      9843
           1      0.699     0.694     0.696       157

    accuracy                          0.991     10000
   macro avg      0.847     0.845     0.846     10000
weighted avg      0.990     0.991     0.990     10000

Confusion matrix:
 [[9796   47]
 [  48  109]]

=== Simulated Real-Time Transaction Stream (first 15 test transactions, sorted to show a mix) ===
  txn_id     amount    score  decision
---------------------------------------------
   34243     133.97    0.874  REVIEW  <-- ACTUAL FRAUD
   23934      54.52    0.067  ALLOW
   18200       9.71    0.057  ALLOW
   15545     314.43    0.887  REVIEW  <-- ACTUAL FRAUD
   33095     179.65    0.076  ALLOW
   24438     117.12    0.883  REVIEW  <-- ACTUAL FRAUD
   3335